# RASP-SFOD — Audit + Train
Run this only after the RASP overlay files are copied into the Drive project.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_PROJECT='/content/drive/MyDrive/MedRT-SFOD'
LOCAL_PROJECT='/content/MedRT-SFOD'
RUNS='/content/drive/MyDrive/MedRT-SFOD-runs'
!rm -rf $LOCAL_PROJECT
!mkdir -p $LOCAL_PROJECT $RUNS
!rsync -a --exclude dataset --exclude runs '$DRIVE_PROJECT/' '$LOCAL_PROJECT/'
%cd $LOCAL_PROJECT
!pip -q install -e .
!pip -q install -r requirements-rasp.txt


In [ ]:
import torch, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('ultralytics', ultralytics.__file__)


## Configure paths


In [ ]:
STAGE1_MODEL = f'{RUNS}/stage1/yolo26m_stage1.pt'  # EDIT
TARGET_YAML = f'{DRIVE_PROJECT}/dataset/c2f_yolo/foggy_cityscapes.yaml'  # EDIT if needed
OUT_DIR = f'{RUNS}/c2f_rasp'
print(STAGE1_MODEL, TARGET_YAML, OUT_DIR, sep='\n')


## Smoke test


In [ ]:
!PYTHONPATH='$PWD' python scripts/YOLO26/test_rasp_smoke.py


## Mandatory pruning-space + DepGraph audit


In [ ]:
!PYTHONPATH='$PWD' python scripts/YOLO26/inspect_rasp_prunable_groups.py \
  --model '$STAGE1_MODEL' --imgsz 256 --device 0 --depgraph \
  --out '$OUT_DIR/rasp_prunable_audit.json'


Inspect the audit output before launching 60 epochs. If the controllable Params/MAC space is tiny, stop and redesign the safe pruning space.


## Train RASP (T4-friendly default batch=4; use 2 if OOM)


In [ ]:
!PYTHONPATH='$PWD' python scripts/YOLO26/stage2_rasp_rtsfod_yolo26.py \
  --stage1_model '$STAGE1_MODEL' --data '$TARGET_YAML' --out_dir '$OUT_DIR' \
  --imgsz 1024 --batch 4 --workers 2 --device 0 --rasp_enable


## Resume after interruption


In [ ]:
# Uncomment to resume
# !PYTHONPATH='$PWD' python scripts/YOLO26/stage2_rasp_rtsfod_yolo26.py \
#   --stage1_model '$STAGE1_MODEL' --data '$TARGET_YAML' --out_dir '$OUT_DIR' \
#   --imgsz 1024 --batch 4 --workers 2 --device 0 --rasp_enable \
#   --resume_state '$OUT_DIR/checkpoints/rasp_training_state_latest.pt'
